# Kaggle League — Submission Notebook

| | |
|---|---|
| **Team name** | *Surname1 & Surname2* |
| **Edition** | 2 (K-nearest neighbours) |
| **Dataset** | regression (`ml1-qf-2027-e2-knn-salary`) |

> **This header cell is required by the Kaggle League rules.** Fill in all three fields. The notebook lives in your team's private repository, in the folder of its competition (e.g. `e1_linear_claims/`), next to the `submission.csv` it produces.

## How to Use This Template

This notebook is a **starting point** that satisfies the formal requirements of the Kaggle League (see the rules in the lecture slides) as long as you keep its structure, fill in the header, point it at the competition files and reproduce the submission you selected:

- a header cell with team, edition and dataset;
- the required packages are listed in the first code cell;
- it loads the **raw** Kaggle files, does all preprocessing and feature engineering, and trains **only models of the edition's family** (an assertion enforces it);
- all random seeds are fixed, so the run is reproducible;
- it produces **exactly** the submission file selected for the private leaderboard, in the format of `sample_submission.csv`;
- it runs top-to-bottom without errors in a clean environment (`uv sync` in the competition's folder).

What it does *not* do is win: the preprocessing is generic (imputation, scaling, one-hot encoding, parsed multi-select columns), the feature engineering is empty, and the hyperparameter grid is tiny. Run as it is, the template lands somewhere between the lecturer's Benchmark I and Benchmark II depending on the edition — the points above that are earned between *Load data* and *Write the submission*. Keep the structure, replace the contents.

**Workflow for every edition** (details in `kaggle_league_template/README.md` of the course repository):

1. Your team works in **one private GitHub repository** created from the course's `kaggle_league_template/` folder and shared with the lecturer. It has one folder per competition (`e1_linear_claims/`, `e1_linear_salary/`, …, `e4_trees_salary/`); each folder is its own uv project and already contains this notebook with `EDITION`, `TASK`, `DATA_DIR` and `SUBMISSION_PATH` set. Run `uv sync` inside the folder.
2. Download `train.csv`, `test.csv` and `sample_submission.csv` from the competition's Data tab into the folder's `data/` subfolder (it is git-ignored — never commit the data).
3. Set `TEAM_NAME`.
4. Develop: features, validation, tuning. Compare every idea with the cross-validated score, **not** with the public leaderboard (30% of the test set — a noisy hint, and you have only 5 submissions per day). The lecturer's reference entries (Baseline, Benchmark I, Benchmark II) are on the leaderboard as unranked benchmark rows, and their public and private scores are on the **Evaluation** tab.
5. For the submission you want to be graded: **restart the kernel and run all cells** — this writes `submission.csv` and prints its MD5 checksum (Section 7). Commit the notebook and `submission.csv`, push, then upload exactly that `submission.csv` to Kaggle with the description `commit <short commit hash> md5 <MD5>` (e.g. `commit 3f2a9c1 md5 9f86d081…`). Kaggle timestamps the description, which proves that the code existed before the deadline.
6. Before the deadline, **select that submission on Kaggle** (one per competition; otherwise Kaggle keeps the best *public* one, which need not be your best *private* one). The lecturer re-runs the notebook at the commit named in its description and compares the MD5.

**This copy is set up for `ml1-qf-2027-e2-knn-salary`.** Put the competition's three files into `data/` next to this notebook (download them from the Data tab of the competition, or with `kaggle competitions download -c ml1-qf-2027-e2-knn-salary -p data` and unzip). Run `uv sync` in this folder, then select the folder's `.venv` as the notebook kernel.

In [ ]:
# Required packages (all part of the course environment: `uv sync`)
# numpy, pandas, scikit-learn, matplotlib
import hashlib
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import sklearn

print(f"pandas {pd.__version__}, numpy {np.__version__}, scikit-learn {sklearn.__version__}")

# 1. Configuration

Everything that changes between editions and datasets lives in this one cell.

In [ ]:
# --- Team and edition ---------------------------------------------------------------------------
TEAM_NAME = "Surname1 & Surname2"
EDITION = 2  # 1: linear/logistic, 2: KNN, 3: SVM, 4: trees & random forest
TASK = "regression"  # "classification" (metric: ROC AUC) or "regression" (metric: RMSE)

# --- Files --------------------------------------------------------------------------------------
# Point this at the folder with the raw Kaggle files: train.csv, test.csv, sample_submission.csv
# (in your team repository: Path("data") inside the competition's folder)
DATA_DIR = Path("data")  # train.csv, test.csv, sample_submission.csv (git-ignored)
SUBMISSION_PATH = Path("submission.csv")  # committed together with this notebook

# --- Reproducibility ----------------------------------------------------------------------------
SEED = 42
np.random.seed(SEED)

# --- Validation, tuning budget and target ------------------------------------------------------
N_FOLDS = 5
TUNE_ROWS = 10_000  # hyperparameter search runs on a stratified subsample of this size (None = all rows); the final fit always uses all rows
# Regression only: fit the model on log1p(target) and transform the predictions back. It often helps the fit of a
# skewed target, but expm1(mean of log) estimates the median rather than the mean, which RMSE penalises - let the
# cross-validated RMSE (Section 6, computed on the original scale) decide.
LOG_TARGET = False

# --- Schema knowledge (not a fitted statistic): which text columns hold semicolon-separated lists ---------------
MULTISELECT_COLUMNS = None  # None = detect the columns whose non-missing values contain ";" (Section 3 prints them)

assert TASK in ("classification", "regression")
assert EDITION in (1, 2, 3, 4)
if TASK == "classification" and LOG_TARGET:
    print("LOG_TARGET is ignored for classification")
print(f"Edition {EDITION} | {TASK} | data: {DATA_DIR.resolve()}")

# 2. Load the Raw Kaggle Files

The submission format is defined by `sample_submission.csv`: its first column is the identifier, its second column is the prediction. The training target has the same name as the prediction column.

In [ ]:
train = pd.read_csv(DATA_DIR / "train.csv")
test = pd.read_csv(DATA_DIR / "test.csv")
sample_submission = pd.read_csv(DATA_DIR / "sample_submission.csv")

ID_COLUMN, PREDICTION_COLUMN = sample_submission.columns[:2]
TARGET_COLUMN = PREDICTION_COLUMN

print(f"train: {train.shape}, test: {test.shape}, sample_submission: {sample_submission.shape}")
print(f"id column: '{ID_COLUMN}', target column: '{TARGET_COLUMN}'")

# Sanity checks on the files
assert TARGET_COLUMN in train.columns, "target column not found in train.csv"
assert TARGET_COLUMN not in test.columns, "test.csv must not contain the target"
assert set(test[ID_COLUMN]) == set(sample_submission[ID_COLUMN]), "test ids and sample_submission ids differ"
assert train[ID_COLUMN].is_unique and test[ID_COLUMN].is_unique, "ids must be unique"
assert sample_submission.shape[1] == 2 and sample_submission[ID_COLUMN].is_unique, (
    "sample_submission must have 2 columns and unique ids"
)

X_train = train.drop(columns=[ID_COLUMN, TARGET_COLUMN])
y_train = train[TARGET_COLUMN]
X_test = test.drop(columns=[ID_COLUMN])

if TASK == "classification":
    assert set(y_train.unique()) == {0, 1}, "classification target must be 0/1"
    print(f"Share of positives: {y_train.mean():.4f}")
else:
    print(
        f"Target: mean {y_train.mean():.4f}, std {y_train.std():.4f}, min {y_train.min():.4f}, max {y_train.max():.4f}"
    )

train.head()

# 3. Quick Look at the Data

Keep this short in the submission notebook — put your real EDA in a separate notebook (see `eda_and_data_wrangling.ipynb` for the full workflow). What must be known before modelling: column types, missing values, cardinality of the categorical columns, and whether train and test look alike.

In [ ]:
numerical_columns = X_train.select_dtypes(include="number").columns.tolist()
categorical_columns = X_train.select_dtypes(exclude="number").columns.tolist()

overview = pd.DataFrame(
    {
        "dtype": X_train.dtypes.astype(str),
        "missing (train)": X_train.isna().mean().round(4),
        "missing (test)": X_test.isna().mean().round(4),
        "n_unique (train)": X_train.nunique(),
    }
)
print(overview)
print(f"\nNumerical features ({len(numerical_columns)}): {numerical_columns}")
print(f"Categorical features ({len(categorical_columns)}): {categorical_columns}")

# Multi-select columns (semicolon-separated lists such as "python;sql") are text, but NOT ordinary categories:
# every combination of items would become its own one-hot category. Section 4 parses them item by item.
# This is schema knowledge, so it is decided here once (not inside the cross-validation).
if MULTISELECT_COLUMNS is None:
    MULTISELECT_COLUMNS = [
        c for c in categorical_columns if X_train[c].dropna().astype(str).str.contains(";").mean() > 0.05
    ]
print(f"Multi-select features ({len(MULTISELECT_COLUMNS)}): {MULTISELECT_COLUMNS}")

# Integer-coded categories (ids, codes) look numerical - check whether a numeric column is really a code
low_cardinality = [c for c in numerical_columns if X_train[c].nunique() <= 20]
print(f"Numerical columns with at most 20 distinct values (codes or counts?): {low_cardinality}")

# Categories present in test but not in train are a classic source of errors at prediction time
for column in [c for c in categorical_columns if c not in MULTISELECT_COLUMNS]:
    unseen = set(X_test[column].dropna()) - set(X_train[column].dropna())
    if unseen:
        print(f"  '{column}': {len(unseen)} categories only in test, e.g. {sorted(map(str, unseen))[:5]}")

# 4. Preprocessing and Feature Engineering

Two rules from the labs:

1. **Stateless** feature engineering — transformations that need *nothing* from the data (ratios, differences, flags, date parts) — goes into a function that is applied to train and test alike.
2. **Stateful** steps — anything that depends on the distribution of the data: imputation values, scaling constants, encodings, bin edges, *and even the decision which columns to transform* — go into transformers inside the model `Pipeline`, so that they are fitted inside every cross-validation fold and never see the validation part. The same fitted objects are then applied to the test set, which guarantees that train and test get identical columns.

Three column types need special care in the League data: **multi-select columns** (semicolon-separated lists) are parsed into an item count and indicator columns of the most frequent items by the `MultiSelectEncoder` below; **ordinal columns** stored as text (education levels, company sizes, age groups) are one-hot encoded here, but an `OrdinalEncoder` with an explicit category order is often the better choice; **mostly-missing columns** keep their information through a missing indicator (the `missing` category for text columns, `add_indicator=True` of the numeric imputer).

Any preprocessing, feature engineering and feature selection is allowed in every edition. This is where most of the points are won.

In [ ]:
from collections import Counter

from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler


def add_features(data):
    """Stateless feature engineering: only transformations that need NOTHING from the data.

    Ratios, differences, flags, date parts - applied identically to train and test, e.g.
        data["debt_to_income"] = data["debt"] / data["income"]
    Anything that depends on the distribution of a column (which columns to log-transform, bin edges,
    scaling constants, imputation values) is NOT stateless: it belongs in a transformer below, so that it
    is fitted on the training part of every cross-validation fold only. Extend freely.
    """
    return data.copy()


class LogSkewed(TransformerMixin, BaseEstimator):
    """Add log1p(x) for the non-negative numeric columns whose skewness exceeds a threshold.

    Which columns qualify is decided in fit() - on the training data of the current fold - and exactly the
    same columns are transformed in transform(), so train, validation and test always get identical features.
    """

    def __init__(self, skew_threshold=2.0):
        self.skew_threshold = skew_threshold

    def fit(self, X, y=None):
        X = pd.DataFrame(X)
        self.feature_names_in_ = np.asarray(X.columns, dtype=object)
        self.columns_ = [c for c in X.columns if X[c].min() >= 0 and X[c].skew() > self.skew_threshold]
        return self

    def transform(self, X):
        X = pd.DataFrame(X).copy()
        for c in self.columns_:
            X[f"{c}_log1p"] = np.log1p(X[c])
        return X

    def get_feature_names_out(self, input_features=None):
        return np.asarray([*self.feature_names_in_, *(f"{c}_log1p" for c in self.columns_)], dtype=object)


class MultiSelectEncoder(TransformerMixin, BaseEstimator):
    """Semicolon-separated multi-select columns -> missing flag, number of items, 0/1 indicator of each frequent item.

    The list of frequent items is learned in fit() (training fold only). Items not in the list are ignored, so the
    output always has the same columns; a missing cell counts as an empty list. The indicators are left as 0/1 (like
    one-hot columns) on purpose: standardising a rare indicator would blow it up to a large value and let it dominate
    the distances of KNN and RBF kernels. `top_k` is a tunable hyperparameter (`preprocessor__multi__items__top_k`).
    """

    def __init__(self, top_k=15, sep=";"):
        self.top_k = top_k
        self.sep = sep

    def _items(self, cell):
        return {item.strip() for item in str(cell).split(self.sep) if item.strip()} if pd.notna(cell) else set()

    def fit(self, X, y=None):
        X = pd.DataFrame(X)
        self.columns_ = list(X.columns)
        self.items_ = {}
        for c in self.columns_:
            counts = Counter(item for cell in X[c] for item in self._items(cell))
            self.items_[c] = [item for item, _ in counts.most_common(self.top_k)]
        return self

    def _feature_names(self):
        names = []
        for c in self.columns_:
            names += [f"{c}__missing", f"{c}__count", *(f"{c}__{item}" for item in self.items_[c])]
        return names

    def transform(self, X):
        X = pd.DataFrame(X)
        columns = []
        for c in self.columns_:
            sets = X[c].map(self._items)
            columns.append(X[c].isna().astype(float).to_numpy())
            columns.append(sets.map(len).to_numpy(dtype=float))
            columns += [sets.map(lambda s, it=item: float(it in s)).to_numpy() for item in self.items_[c]]
        return np.column_stack(columns)  # same order as _feature_names()

    def get_feature_names_out(self, input_features=None):
        return np.asarray(self._feature_names(), dtype=object)


X_train_fe = add_features(X_train)
X_test_fe = add_features(X_test)

numerical_columns = X_train_fe.select_dtypes(include="number").columns.tolist()
multiselect_columns = [c for c in MULTISELECT_COLUMNS if c in X_train_fe.columns]
categorical_columns = [c for c in X_train_fe.select_dtypes(exclude="number").columns if c not in multiselect_columns]

preprocessor = ColumnTransformer(
    transformers=[
        (
            "num",
            Pipeline(
                [
                    ("log", LogSkewed()),  # data-dependent choice of columns -> fitted inside the pipeline
                    ("impute", SimpleImputer(strategy="median", add_indicator=True)),  # keeps "was missing" as a column
                    ("scale", StandardScaler()),
                ]
            ),
            numerical_columns,
        ),
        (
            "cat",
            Pipeline(
                [
                    ("impute", SimpleImputer(strategy="constant", fill_value="missing")),
                    ("onehot", OneHotEncoder(handle_unknown="infrequent_if_exist", min_frequency=0.01)),
                ]
            ),
            categorical_columns,
        ),
        (
            "multi",
            Pipeline([("items", MultiSelectEncoder(top_k=15))]),
            multiselect_columns,
        ),  # no scaling: 0/1 like one-hot
    ],
    verbose_feature_names_out=False,
)

print(
    f"Input features: {X_train_fe.shape[1]} ({len(numerical_columns)} numerical, {len(categorical_columns)} categorical, "
    f"{len(multiselect_columns)} multi-select)"
)

# 5. The Model — Restricted to the Edition's Family

`ALLOWED_MODELS` mirrors the *allowed models* table of the rules, as import paths. Only the classes of the **current edition** are imported — this notebook never touches any other model family. The assertion makes it impossible to submit predictions from a model outside the family by accident. Ensembles of models from the same family are allowed; anything else that *produces the submitted predictions* is not — including predictions carried over from a previous edition.

The assertion looks *through* wrappers and ensembles: `TransformedTargetRegressor` (regression with `LOG_TARGET = True`: the model is fitted on `log1p(y)` and its predictions are transformed back), and same-family ensembles such as `VotingRegressor` / `VotingClassifier`, `BaggingRegressor` / `BaggingClassifier` or `StackingClassifier` whose members — and final estimator — are all from the edition's family. For `SGDClassifier` / `SGDRegressor` only the logistic / squared-error losses count as edition 1 (a hinge loss is an SVM). One scoring rule: for classifiers without `predict_proba` such as `SVC`, the **decision function** is the submitted score — ROC AUC only depends on the *ranking* of the rows, so a monotone score is as good as a probability (and much cheaper than `CalibratedClassifierCV` on tens of thousands of rows).

Replace the default model with your tuned one — but keep the assertion.

In [ ]:
import importlib

from scipy.stats import rankdata
from sklearn.compose import TransformedTargetRegressor

# The allowed-models table of the rules (one family per edition), as import paths - strings only.
ALLOWED_MODELS = {
    1: (
        "sklearn.linear_model.LinearRegression",
        "sklearn.linear_model.Ridge",
        "sklearn.linear_model.Lasso",
        "sklearn.linear_model.ElasticNet",
        "sklearn.linear_model.LogisticRegression",
        "sklearn.linear_model.SGDRegressor",
        "sklearn.linear_model.SGDClassifier",
    ),
    2: ("sklearn.neighbors.KNeighborsClassifier", "sklearn.neighbors.KNeighborsRegressor"),
    3: (
        "sklearn.svm.SVC",
        "sklearn.svm.NuSVC",
        "sklearn.svm.LinearSVC",
        "sklearn.svm.SVR",
        "sklearn.svm.NuSVR",
        "sklearn.svm.LinearSVR",
    ),
    4: (
        "sklearn.tree.DecisionTreeClassifier",
        "sklearn.tree.DecisionTreeRegressor",
        "sklearn.ensemble.RandomForestClassifier",
        "sklearn.ensemble.RandomForestRegressor",
        "sklearn.ensemble.ExtraTreesClassifier",
        "sklearn.ensemble.ExtraTreesRegressor",
    ),
}

# Library-default estimator of every (edition, task) - the starting point. Tune it with cross-validation!
DEFAULT_MODELS = {
    (1, "classification"): ("sklearn.linear_model.LogisticRegression", {"max_iter": 5000}),
    (1, "regression"): ("sklearn.linear_model.Ridge", {}),
    (2, "classification"): ("sklearn.neighbors.KNeighborsClassifier", {}),
    (2, "regression"): ("sklearn.neighbors.KNeighborsRegressor", {}),
    (3, "classification"): ("sklearn.svm.SVC", {}),
    (3, "regression"): ("sklearn.svm.SVR", {}),
    (4, "classification"): (
        "sklearn.ensemble.RandomForestClassifier",
        {"n_estimators": 300, "random_state": SEED, "n_jobs": -1},
    ),
    (4, "regression"): (
        "sklearn.ensemble.RandomForestRegressor",
        {"n_estimators": 300, "random_state": SEED, "n_jobs": -1},
    ),
}


def load_class(import_path):
    """Import a class from its dotted path, e.g. 'sklearn.linear_model.Ridge'."""
    module_name, class_name = import_path.rsplit(".", 1)
    return getattr(importlib.import_module(module_name), class_name)


def core_estimator(model):
    """The estimator that actually produces the predictions (unwrapping the allowed target-transform wrapper)."""
    return model.regressor if isinstance(model, TransformedTargetRegressor) else model


def leaf_estimators(model):
    """All estimators that contribute predictions: looks through wrappers and same-family ensembles."""
    children = []
    for attribute in ("regressor", "estimator", "final_estimator"):
        child = getattr(model, attribute, None)
        if hasattr(child, "fit"):
            children.append(child)
    for item in getattr(model, "estimators", None) or []:
        children.append(item[1] if isinstance(item, tuple) else item)
    return [leaf for child in children for leaf in leaf_estimators(child)] if children else [model]


def check_allowed(model):
    """The rule check: every estimator that produces predictions must belong to the edition's family.

    It inspects the model step of the pipeline (wrappers and ensembles included) - it cannot see predictions
    that were produced elsewhere, so the rule itself is still your responsibility.
    """
    for leaf in leaf_estimators(model):
        assert isinstance(leaf, allowed_classes), f"{type(leaf).__name__} is not allowed in edition {EDITION}"
        if type(leaf).__name__ == "SGDClassifier":
            assert leaf.loss == "log_loss", "edition 1: SGDClassifier must use loss='log_loss' (hinge = SVM)"
        if type(leaf).__name__ == "SGDRegressor":
            assert leaf.loss == "squared_error", (
                "edition 1: SGDRegressor must use loss='squared_error' (epsilon-insensitive = SVR)"
            )


def predict_scores(fitted_pipeline, X):
    """Submitted predictions: probabilities of class 1 (or a monotone score) for classification, values for regression."""
    if TASK == "regression":
        return fitted_pipeline.predict(X)
    if hasattr(fitted_pipeline, "predict_proba"):
        return fitted_pipeline.predict_proba(X)[:, 1]  # probabilities of class 1 - never predict()!
    # e.g. SVC: the decision function is a monotone score, so ROC AUC is unchanged by any strictly increasing map.
    # Ranks (scaled into (0, 1]) keep the ordering exactly; a sigmoid would saturate large margins into ties.
    decision = fitted_pipeline.decision_function(X)
    return rankdata(decision) / len(decision)


# Only the current edition's family is imported
allowed_classes = tuple(load_class(path) for path in ALLOWED_MODELS[EDITION])
default_path, default_params = DEFAULT_MODELS[(EDITION, TASK)]
model = load_class(default_path)(**default_params)

# Regression with a log-transformed target: the wrapper fits on log1p(y) and predicts expm1(.) back
if TASK == "regression" and LOG_TARGET:
    assert (y_train > -1).all(), "LOG_TARGET needs a target above -1 (log1p); shift or disable it"
    model = TransformedTargetRegressor(regressor=model, func=np.log1p, inverse_func=np.expm1)
    PARAM_PREFIX = "model__regressor__"  # hyperparameter names in the pipeline, e.g. model__regressor__alpha
else:
    PARAM_PREFIX = "model__"  # e.g. model__C

check_allowed(model)
if EDITION == 3 and TASK == "regression" and not LOG_TARGET:
    print("Note: SVR with default C on a target in the tens of thousands is almost a constant predictor -")
    print("      the example grid therefore uses large C values; LOG_TARGET = True is the other way out.")

pipeline = Pipeline([("preprocessor", preprocessor), ("model", model)])
print(f"Edition {EDITION} ({TASK}) default model: {model}")
pipeline

# 6. Validation

The metric of the competition is **ROC AUC** (classification — submit *probabilities* of class 1 or another monotone score, never hard labels) or **RMSE** (regression, on the original scale of the target). Cross-validation on the training set is your leaderboard during development:

- it uses all the training data,
- it is far less noisy than the public leaderboard (the claims public board contains only about 80 claims, so its ROC AUC has a standard error of roughly 0.03 — public differences below 0.05 are noise; a public RMSE moves by a few per cent with the large salaries that happen to be in it), and it does not consume submissions,
- the folds are fixed (`cv` below), so compare two ideas by their **paired per-fold differences**; the fold spread is a rough stability diagnostic, not a significance test.

We also compute the score of the **naive baseline** (a constant prediction) — the first bar to clear in the scoring table. The lecturer's three reference scores for the edition (naive baseline, Benchmark I = default model on the raw features, Benchmark II = tuned model on engineered features) are published on the **Evaluation** tab of the competition, public and private: beating the baseline on the private leaderboard gives 2 points, beating Benchmark I 4 points, beating Benchmark II 5 points (the highest bar you clear counts, not the sum).

In [ ]:
from sklearn.dummy import DummyClassifier, DummyRegressor
from sklearn.model_selection import KFold, StratifiedKFold, cross_val_score

if TASK == "classification":
    cv = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
    scoring = "roc_auc"
    baseline = DummyClassifier(strategy="prior")
else:
    cv = KFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
    scoring = "neg_root_mean_squared_error"
    baseline = DummyRegressor(strategy="mean")


def report(name, scores):
    sign = 1 if TASK == "classification" else -1
    print(
        f"{name:<42s} {scoring:>28s}: {sign * scores.mean():.4f} (± {scores.std():.4f})  folds: {np.round(sign * scores, 4)}"
    )


with warnings.catch_warnings():
    warnings.simplefilter("ignore")  # a constant score is a degenerate ROC curve; scikit-learn warns and returns 0.5
    baseline_scores = cross_val_score(baseline, X_train_fe, y_train, cv=cv, scoring=scoring)
model_scores = cross_val_score(pipeline, X_train_fe, y_train, cv=cv, scoring=scoring, n_jobs=-1)

print("=" * 100)
print(f"{N_FOLDS}-FOLD CROSS-VALIDATION ON THE TRAINING SET")
print("=" * 100)
report("Naive baseline (constant)", baseline_scores)
report(f"{type(core_estimator(model)).__name__} (this notebook)", model_scores)

## Runtime Notes for the Larger Dataset (KNN, SVM)

The claims data have about 39,000 training rows; the salary data about 8,700 rows with a few hundred columns after parsing the multi-select features. Two families need care:

- **SVM (edition 3).** Kernel `SVC` / `SVR` scale roughly quadratically with the number of rows: one fit on 39,000 rows takes minutes, a 5-fold grid on all rows takes hours — hence the `TUNE_ROWS` subsample used by the search below. Try `LinearSVC` / `LinearSVR` (seconds) first; keep `probability=False` and submit the decision function (Section 5). For `SVR`, the default `C=1, epsilon=0.1` on a target in the tens of thousands is close to a constant prediction — raise `C` (and `epsilon`, which is in target units) by orders of magnitude as the example grid does, or change the target: `LOG_TARGET = True` is a different objective (log errors) that may or may not help the USD-scale RMSE, and standardising the target inside `TransformedTargetRegressor(transformer=StandardScaler())` is the pure scaling alternative.
- **KNN (edition 2).** With 1.7% positives and `k=5`, `predict_proba` takes only six distinct values and most rows tie at 0 — a poor ROC AUC. Use a large `k` (tens to hundreds), `weights="distance"`, and think about how much each block of features should weigh in the distance: a one-hot block contributes 0 or 2 to the squared Euclidean distance whatever its cardinality, a standardised numeric column contributes on its own scale, and a standardised *rare* 0/1 indicator would contribute a huge amount for the few rows that have it (which is why the multi-select indicators are left as 0/1). Feature selection and per-block weights are the levers.
- **Everything:** `n_jobs=-1` in the searches, and develop on a subsample before running the final configuration on all data.

## Hyperparameter Tuning (Your Work Starts Here)

The cell below is a minimal example of a cross-validated search with the pipeline. Extend the grid, switch to `RandomizedSearchCV`, add feature selection steps — see the lab notebook of the edition's model family. Remember: `best_score_` is optimistically biased (it is the maximum over the candidates), and re-running `cross_val_score` on the winner with the same folds does not remove that bias. For an honest estimate after a large search use nested cross-validation or a validation set that the search never saw (`core_ml_techniques.ipynb`); for everyday development, comparing ideas by ordinary CV is fine.

**Budget.** The search runs on a stratified subsample of `TUNE_ROWS` rows (configuration cell) when the training set is larger; the chosen configuration is then refitted on all rows in Section 7. On the claims data (about 39,000 rows) this is what makes a kernel SVM search feasible: one `SVC` fit on all rows takes minutes, so a 5-fold grid on all rows takes hours. Note that a subsample keeps the class ratio; if you subsample *negatives only* to speed things up further, the best `C` / `class_weight` can shift when you refit on the full data — re-check on a second subsample.

In [ ]:
from sklearn.model_selection import GridSearchCV, train_test_split

# Example grids per edition - deliberately tiny so that the template runs in seconds. Replace them.
# PARAM_PREFIX is "model__" or, with LOG_TARGET, "model__regressor__" (the model sits inside the wrapper).
P = PARAM_PREFIX
EXAMPLE_GRIDS = {
    1: {f"{P}C": [0.1, 1.0, 10.0]} if TASK == "classification" else {f"{P}alpha": [0.1, 1.0, 10.0]},
    2: {f"{P}n_neighbors": [15, 51, 151], f"{P}weights": ["uniform", "distance"]},
    # SVR on an unscaled target in the tens of thousands needs a C of the same order of magnitude
    3: {f"{P}C": [0.3, 1.0, 3.0]} if TASK == "classification" or LOG_TARGET else {f"{P}C": [1e3, 1e4, 1e5]},
    4: {f"{P}max_features": ["sqrt", 0.5], f"{P}min_samples_leaf": [1, 5]},
}

# Tune on a stratified subsample when the training set is large (the final fit in Section 7 uses all rows)
if TUNE_ROWS is not None and len(X_train_fe) > TUNE_ROWS:
    strata = y_train if TASK == "classification" else pd.qcut(y_train, 10, labels=False, duplicates="drop")
    X_tune, _, y_tune, _ = train_test_split(
        X_train_fe, y_train, train_size=TUNE_ROWS, stratify=strata, random_state=SEED
    )
    print(f"Tuning on a stratified subsample of {len(X_tune):,} of {len(X_train_fe):,} rows")
else:
    X_tune, y_tune = X_train_fe, y_train

search = GridSearchCV(pipeline, EXAMPLE_GRIDS[EDITION], cv=cv, scoring=scoring, n_jobs=-1)
search.fit(X_tune, y_tune)

sign = 1 if TASK == "classification" else -1
print(f"Best parameters: {search.best_params_}")
print(
    f"Best CV {scoring}: {sign * search.best_score_:.4f}  (selection-biased - confirm with cross_val_score if it matters)"
)

# Use the tuned configuration from here on (an unfitted copy: Section 7 fits it on ALL training rows)
final_pipeline = search.best_estimator_
check_allowed(final_pipeline.named_steps["model"])

# 7. Train on All Data and Write the Submission

The final model is fitted on the **whole** training set, then applied to the test set. The submission file must have exactly the columns and the row order of `sample_submission.csv`.

The cell also prints the **MD5 checksum** of the file. The lecturer re-runs the notebook at the commit named in your Kaggle submission description and compares the checksum with the one in that description (on another machine tiny floating-point differences can appear; the check then falls back to comparing the predictions themselves and the private score).

In [ ]:
final_pipeline.fit(X_train_fe, y_train)
predictions = predict_scores(final_pipeline, X_test_fe)

submission = pd.DataFrame({ID_COLUMN: test[ID_COLUMN], PREDICTION_COLUMN: predictions})

# Format checks against sample_submission
assert list(submission.columns) == list(sample_submission.columns[:2])
assert len(submission) == len(sample_submission)
assert submission[ID_COLUMN].tolist() == sample_submission[ID_COLUMN].tolist(), "row order must match sample_submission"
assert np.isfinite(submission[PREDICTION_COLUMN]).all(), "predictions contain NaN or inf"
if TASK == "classification":
    assert submission[PREDICTION_COLUMN].between(0, 1).all(), "probabilities must lie in [0, 1]"

SUBMISSION_PATH.parent.mkdir(parents=True, exist_ok=True)
submission.to_csv(SUBMISSION_PATH, index=False, float_format="%.10g")  # enough digits to avoid artificial ties

checksum = hashlib.md5(SUBMISSION_PATH.read_bytes()).hexdigest()
print(f"Submission written to: {SUBMISSION_PATH}")
print(f"Rows: {len(submission)}, MD5: {checksum}")
print(
    "Commit this notebook and the file, push, and use the Kaggle submission description: commit <short hash> md5 "
    + checksum
)
print("\nFirst rows:")
print(submission.head())

fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(predictions, bins=40, edgecolor="black", alpha=0.75)
ax.set_title("Distribution of the Submitted Predictions", fontsize=13, fontweight="bold")
ax.set_xlabel(PREDICTION_COLUMN)
plt.tight_layout()
plt.show()

# 8. Pre-Submission Checklist

- [ ] Header cell filled in: team name, edition, dataset; `TEAM_NAME` set in the configuration cell
- [ ] The notebook is in its competition's folder of your team repository; `EDITION` and `TASK` match the folder name
- [ ] The notebook loads the raw Kaggle files from `data/` only — no external data (in particular not the public sources the datasets were derived from, nor files of previous course editions), no hand-labelled test rows; `data/` is not committed
- [ ] All predictions come from a model of the edition's family (the assertions passed); other models were used at most for auxiliary tasks (e.g. feature selection); nothing is carried over from a previous edition
- [ ] `Kernel → Restart Kernel and Run All Cells` completes without errors after `uv sync` in the folder
- [ ] Notebook and `submission.csv` committed and pushed **before** uploading the file to Kaggle; the Kaggle description reads `commit <hash> md5 <MD5>` with this run's MD5
- [ ] That submission is **selected** on Kaggle before the deadline (Sunday 23:59, Warsaw time) — one per competition
- [ ] The notebook is your team's own work (public tutorials as a source of ideas are fine; copied solutions are not); the repository stays private until the League has ended